# yolox_cpp — real-GPU check (Colab)

Builds the from-scratch pure engine with `nvcc -DUSE_CUDA` and runs YOLOX on a real
GPU. Same source runs on CPU with g++. **Runtime → Change runtime type → GPU (T4).**


### 1. GPU + nvcc


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -2


### 2. Clone


In [ ]:
%cd /content
!rm -rf yolox_cpp
!git clone -q https://github.com/yomei-o/yolox_cpp.git
%cd /content/yolox_cpp


### 3. GPU inference demo (shipped weights, no Python)
Weights ship in `weights/yolox_tiny/`, so the detector runs straight from the clone.


In [ ]:
!nvcc -x cu -std=c++17 --extended-lambda -DUSE_CUDA -O2 -Ipure/third_party pure/m_demo.cpp -o m_demo 2>&1 | tail -15
!./m_demo assets/bus.jpg out.png 640


### 4. Weights + reference (Ultralytics / torch.hub YOLOX)


In [ ]:
!pip -q install ultralytics==8.4.104 loguru tabulate thop >/dev/null 2>&1
!python pure/ref/export_yolox.py 64 yolox_tiny 2>/dev/null | tail -1


### 5. Full forward on the GPU vs PyTorch


In [ ]:
!nvcc -x cu -std=c++17 --extended-lambda -DUSE_CUDA -O2 pure/m1_forward.cpp -o m1 2>&1 | tail -15
!./m1 2>&1 | tail -5


### 6. Train on the GPU (SimOTA + IoU/BCE, all CUDA)


In [ ]:
!nvcc -x cu -std=c++17 --extended-lambda -DUSE_CUDA -O2 pure/m3_train.cpp -o train 2>&1 | tail -15
!./train 2>&1 | tail -10


### Success
- Cell 3: `bus` + people detected → inference on real CUDA.
- Cell 5: `yolox M1: PURE ENGINE == yolox-tiny forward`.
- Cell 6: loss drops → training runs on the GPU.

Change `yolox_tiny`→`yolox_s/m/l/x/nano` in cell 4 for other sizes.
